In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

import plotly.graph_objects as go

## Linear regression: defining the line

In [ ]:
def standard_units(x):
    """Converts an array x to standard units"""
    return (x - np.mean(x)) / np.std(x)

In [ ]:
def correlation(t, x, y):
    x_su = standard_units(t.column(x))
    y_su = standard_units(t.column(y))
    return np.mean(x_su * y_su)

In [ ]:
def slope(t, x, y):
    """t is a table, x and y are column labels, 
    returns slope of least squares regression line"""
    r = correlation(t, x, y)
    slope_xy = r * np.std(t.column(y)) / np.std(t.column(x))
    return slope_xy

In [ ]:
def intercept(t, x, y):
    """t is a table, x and y are column labels, 
    returns intercept of least squares regression line"""
    slope_xy = slope(t, x, y)
    intercept_xy = np.average(t.column(y)) - slope_xy * np.average(t.column(x))
    return intercept_xy

## Prediction

In [ ]:
demographics = Table.read_table('/home/jovyan/shared/Data/district_demographics2016.csv')
demographics.show(5)

In [ ]:
demographics = demographics.select('College%', 'Median Income')
demographics.show(5)

In [ ]:
regression_slope = ...
regression_intercept = ...
regression_slope, regression_intercept

In [ ]:
demographics.scatter('College%', 'Median Income', fit_line = True)

In [ ]:
# For 50% College 



In [ ]:
demographics = demographics.with_column(
    'Linear Prediction', 
    fitted_values(demographics, 'College%', 'Median Income'))

demographics.scatter('College%')

## Errors

In [ ]:
errors = ...

In [ ]:
demographics = demographics.with_column('Error', errors)
demographics

## Numerical Optimization ###

In [ ]:
x = np.arange(1, 3, 0.1)
y = (x-2)**2 + 3
Table().with_columns('x', x, 'y', y).plot('x')

In [ ]:
x = np.arange(-1.5, 1.5, 0.05)
y2 = 2 * np.sin(x*np.pi) + x ** 3 + x ** 4 
Table().with_columns('x', x, 'y', y2).plot('x')

### Minimizing RMSE ###

In [ ]:
best_slope = slope(demographics, 'College%', 'Median Income')
best_intercept = intercept(demographics, 'College%', 'Median Income')

slopes = np.arange(best_slope - 1000, best_slope + 1000, 20)

intercepts = np.arange(best_intercept - 20000, best_intercept + 20000, 400)

rmse_values = make_array()

for any_intercept in intercepts:
    row = make_array()

    for any_slope in slopes:
        row = np.append(row, demographics_rmse(any_slope, any_intercept))

    rmse_values = np.append(rmse_values, row)

rmse_values = rmse_values.reshape(intercepts.size, slopes.size)

minimum_rmse = demographics_rmse(best_slope, best_intercept)

fig = go.Figure()

fig.add_trace(
    go.Surface(x = slopes, y = intercepts, z = rmse_values))

fig.add_trace(
    go.Scatter3d(
        x = make_array(best_slope),
        y = make_array(best_intercept),
        z = make_array(minimum_rmse)))


fig.update_layout(
    title='Minimizing the Standard Deviation of Errors',
    scene=dict(
        xaxis_title='Slope',
        yaxis_title='Intercept',
        zaxis_title='RMSE'
    ),
    width=1000,
    height=750
)